In [0]:
pip install faker
%restart_python

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
"""
Funções para geração de dados sintéticos de uma transportadora logística:
armazéns, clientes, produtos, funcionários e vendas — além do cálculo de
distâncias reais (via OpenRouteService) e roteirização simples por
vizinho mais próximo.
 
Observação: o projeto é uma homenagem à sitcom "The Office" (versão
americana) — os funcionários em monta_funcionarios() são personagens da
Dunder Mifflin (Jim Halpert, Dwight Schrute, Michael Scott, etc.), com
municípios brasileiros usados apenas para dar realismo geográfico aos
dados sintéticos (endereços, distâncias, rotas).
"""

import io
import random

import pandas as pd
import requests
from faker import Faker

# ---------------------------------------------------------------------
# Configuração inicial
# ---------------------------------------------------------------------

fake = Faker('pt_BR')
Faker.seed(42)
random.seed(42)  # garante reprodutibilidade também para random.choice/sample,
                  # não só para o Faker

URL_IBGE = "https://servicodados.ibge.gov.br/api/v1/localidades/municipios"
URL_LAT_LONG = "https://raw.githubusercontent.com/kelvins/municipios-brasileiros/main/csv/municipios.csv"


# ---------------------------------------------------------------------
# Municípios e geolocalização
# ---------------------------------------------------------------------

def carrega_municipios():
    """
    Busca todos os municípios do Brasil na API do IBGE e retorna um
    DataFrame já achatado (sem colunas aninhadas), com o nome, UF,
    código da UF e região de cada um.
    """
    response = requests.get(URL_IBGE)
    response.raise_for_status()
    dados_json = response.json()

    df = pd.json_normalize(dados_json)
    df = df[[
        "id",
        "nome",
        "microrregiao.mesorregiao.UF.sigla",
        "microrregiao.mesorregiao.UF.id",
        "microrregiao.mesorregiao.UF.regiao.nome",
    ]]
    df.columns = ["codigo_ibge", "nome_cidade", "uf", "sigla_id", "regiao"]
    return df


def carrega_lat_long():
    """
    Baixa o CSV com latitude/longitude de todos os municípios brasileiros
    (fonte: kelvins/municipios-brasileiros) e retorna só as colunas
    necessárias para geolocalização.

    Usa um User-Agent customizado porque o GitHub às vezes bloqueia (403)
    requisições sem esse cabeçalho.
    """
    resp = requests.get(URL_LAT_LONG, headers={'User-Agent': 'Mozilla/5.0'})
    resp.raise_for_status()
    df = pd.read_csv(io.StringIO(resp.text))
    return df[['nome', 'codigo_uf', 'latitude', 'longitude']]


def buscar_lat_long(df_lat_long, nome_cidade, codigo_uf):
    """
    Retorna (latitude, longitude) de uma cidade específica.

    Filtra por nome E código da UF ao mesmo tempo — filtrar só por nome
    é arriscado, pois existem cidades homônimas em estados diferentes
    (ex: "Belém" existe no PA, MG e AL).
    """
    linha = df_lat_long[
        (df_lat_long['nome'] == nome_cidade) & (df_lat_long['codigo_uf'] == codigo_uf)
    ].head(1)

    if linha.empty:
        raise ValueError(f"Cidade não encontrada em df_lat_long: {nome_cidade} / UF {codigo_uf}")

    return linha['latitude'].values[0], linha['longitude'].values[0]


# ---------------------------------------------------------------------
# Entidades de negócio
# ---------------------------------------------------------------------

def montar_armazens(df_municipios, df_lat_long):
    """
    Monta os armazéns fixos da transportadora, um em cada cidade da lista
    abaixo, com código, localização e coordenadas.

    Recebe df_municipios e df_lat_long como parâmetros (em vez de buscar
    internamente) para evitar repetir chamadas de API desnecessárias
    quando montar_clientes() também precisar dos mesmos dados.
    """
    armazem_cidades = ['São Paulo', 'Curitiba', 'Recife', 'Belém']
    armazem_codigo = ['A1', 'A2', 'A3', 'A4']

    armazens = []
    for i, cidade in enumerate(armazem_cidades):
        linha = df_municipios[df_municipios['nome_cidade'] == cidade].head(1)

        if linha.empty:
            raise ValueError(f"Cidade não encontrada em df_municipios: {cidade}")

        codigo_uf = linha['sigla_id'].values[0]
        uf = linha['uf'].values[0]
        regiao = linha['regiao'].values[0]
        latitude, longitude = buscar_lat_long(df_lat_long, cidade, codigo_uf)

        armazens.append({
            'id': i + 1,
            'codigo': armazem_codigo[i],
            'cidade': cidade,
            'uf': uf,
            'regiao': regiao,
            'latitude': latitude,
            'longitude': longitude,
        })
    return pd.DataFrame(armazens)


def montar_produtos():
    """Catálogo fixo de produtos (papelaria) com peso e valor unitário."""
    produtos = [
        [1, 'Papel Kraft', 3, 80],
        [2, 'Papel-Cartão', 20, 250],
        [3, 'Papel Supremo', 20, 220],
        [4, 'Papel Sulfite', 23, 300],
        [5, 'Papel Couché', 6, 200],
        [6, 'Papel Pólen', 13, 220],
        [7, 'Papel Color Plus', 2.8, 100],
        [8, 'Papel Vegetal', 2, 80],
        [9, 'Papel Fotográfico', 1.2, 40],
    ]
    return pd.DataFrame(produtos, columns=['id', 'produto', 'peso', 'valor_unitario'])


def montar_clientes(df_municipios, df_lat_long, quantidade=20):
    """
    Sorteia `quantidade` municípios aleatórios e gera um cliente fictício
    (empresa) para cada um, com dados de contato via Faker.
    """
    clientes = []
    for x in range(quantidade):
        linha_aleatoria = df_municipios.sample(n=1)
        nome_cidade = linha_aleatoria['nome_cidade'].values[0]
        codigo_uf = linha_aleatoria['sigla_id'].values[0]
        regiao = linha_aleatoria['regiao'].values[0]
        uf = linha_aleatoria['uf'].values[0]

        latitude, longitude = buscar_lat_long(df_lat_long, nome_cidade, codigo_uf)

        clientes.append({
            'id': x + 1,
            'name': fake.company(),
            'endereco_entrega': fake.street_address(),
            'cidade': nome_cidade,
            'uf': uf,
            'regiao': regiao,
            'latitude': latitude,
            'longitude': longitude,
            'email': fake.ascii_company_email(),
            'telefone': fake.phone_number(),
        })
    return pd.DataFrame(clientes)


def monta_funcionarios():
    """Quadro fixo de funcionários, cada um com uma área e data de contratação aleatória."""
    funcionarios = [
        [2, 'Jim Halpert', 'Vendas'],
        [1, 'Dwight Schrute', 'Vendas'],
        [3, 'Stanley Hudson', 'Vendas'],
        [4, 'Phyllis Vance', 'Vendas'],
        [5, 'Andy Bernard', 'Vendas'],
        [6, 'Ryan Howard', 'Vendas'],
        [7, 'Michael Scott', 'Gerencia'],
        [8, 'Pamela Halpert', 'Administrativo'],
        [9, 'Angela Martin', 'Contabilidade'],
        [10, 'Oscar Martinez', 'Contabilidade'],
        [11, 'Kevin Malone', 'Contabilidade'],
        [12, 'Kelly Kapoor', 'Administrativo'],
        [13, 'Creed Bratton', 'Administrativo'],
        [14, 'Meredith Palmer', 'Administrativo'],
        [15, 'Darryl Philbin', 'Armazém'],
        [16, 'Lonny Smith', 'Armazém'],
        [17, 'Madge Parker', 'Armazém'],
        [18, 'Glenn Godwin', 'Armazém'],
        [19, 'Hide Lee', 'Armazém'],
        [20, 'Toby Flenderson', 'Recursos Humanos'],
    ]
    datas_possiveis = pd.date_range(start="2020-01-01", end="2022-05-10")

    registros = []
    for id_func, nome, area in funcionarios:
        registros.append({
            'id': id_func,
            'Nome': nome,
            'Area': area,
            'Data_Contratacao': random.choice(datas_possiveis),
        })
    return pd.DataFrame(registros)


def montar_vendas(df_produtos, df_clientes, df_funcionarios, datas, n):
    """
    Gera `n` vendas sintéticas, sorteando produto, cliente, vendedor
    (apenas funcionários da área "Vendas") e data.
    """
    # calculado uma única vez fora do loop, já que não muda a cada venda
    vendedores = df_funcionarios[df_funcionarios['Area'] == 'Vendas']['id'].tolist()

    vendas = []
    for x in range(n):
        produto = df_produtos.sample(n=1).iloc[0]
        qtd = random.randint(1, 30)

        vendas.append({
            'id_venda': x + 1,
            'data_venda': random.choice(datas),
            'id_cliente': random.choice(df_clientes['id']),
            'id_vendedor': random.choice(vendedores),
            'produto_id': produto['id'],
            'quantidade': qtd,
            'valor_total': qtd * produto['valor_unitario'],
        })
    return pd.DataFrame(vendas)


# ---------------------------------------------------------------------
# Distâncias (OpenRouteService) e atribuição de armazém
# ---------------------------------------------------------------------

def calcula_distancias(df_armazem, df_clientes, api_key):
    """
    Calcula a distância rodoviária entre cada armazém e cada cliente
    via OpenRouteService Matrix API, em uma única chamada.

    Usa "sources"/"destinations" para pedir apenas a matriz armazém→cliente,
    evitando gastar cota da API com pares armazém↔armazém ou cliente↔cliente.
    """
    locations_armazem = df_armazem[['longitude', 'latitude']].values.tolist()
    locations_cliente = df_clientes[['longitude', 'latitude']].values.tolist()
    locations = locations_armazem + locations_cliente

    n_armazens = len(locations_armazem)
    n_clientes = len(locations_cliente)

    url_matrix = "https://api.openrouteservice.org/v2/matrix/driving-car"
    headers = {
        'Authorization': api_key,
        'Content-Type': 'application/json',
    }
    body = {
        "locations": locations,
        "sources": list(range(n_armazens)),
        "destinations": list(range(n_armazens, n_armazens + n_clientes)),
        "metrics": ["distance"],
    }

    response = requests.post(url_matrix, json=body, headers=headers)
    dados = response.json()

    if 'distances' not in dados:
        raise RuntimeError(f"Erro na API do OpenRouteService: {dados}")

    registros_distancia = []
    for i in range(n_armazens):
        for j in range(n_clientes):
            registros_distancia.append({
                'id_armazem': df_armazem.iloc[i]['id'],
                'id_cliente': df_clientes.iloc[j]['id'],
                'distancia_metros': dados['distances'][i][j],
            })
    return pd.DataFrame(registros_distancia)


def atribui_armazem(df_armazem, df_clientes, api_key):
    """
    Para cada cliente, encontra o armazém mais próximo.

    Faz o cross join armazém × cliente, junta com as distâncias calculadas
    e mantém, por cliente, apenas a linha com a menor distância.
    """
    df_distancias = calcula_distancias(df_armazem, df_clientes, api_key)

    df = df_armazem.merge(
        df_clientes,
        how='cross',
        suffixes=('_armazem', '_cliente'),
    )
    df = df.merge(
        df_distancias,
        on=['id_armazem', 'id_cliente'],
    ).drop(
        columns=[
            'codigo', 'cidade_armazem', 'latitude_armazem', 'longitude_armazem',
            'name', 'endereco_entrega', 'cidade_cliente', 'uf_cliente',
            'latitude_cliente', 'longitude_cliente', 'email', 'telefone',
        ]
    ).reset_index(drop=True)

    df['distancia_km'] = (df['distancia_metros'] / 1000).round(2)
    df = df.drop(columns=['distancia_metros'])

    # mantém só a linha de menor distância para cada cliente
    indice_menor_distancia = df.groupby('id_cliente')['distancia_km'].idxmin()
    return df.loc[indice_menor_distancia]


def calcula_distancias_clientes(df_clientes, api_key):
    """
    Calcula a distância rodoviária entre cada par de clientes
    (matriz cliente × cliente completa, excluindo a diagonal
    onde origem == destino).

    Necessário para a roteirização: depois da primeira entrega, o
    caminho segue de cliente em cliente, não volta ao armazém.
    """
    locations = df_clientes[['longitude', 'latitude']].values.tolist()
    n = len(locations)

    url_matrix = "https://api.openrouteservice.org/v2/matrix/driving-car"
    headers = {
        'Authorization': api_key,
        'Content-Type': 'application/json',
    }
    body = {
        "locations": locations,
        "metrics": ["distance"],
    }

    response = requests.post(url_matrix, json=body, headers=headers)
    dados = response.json()

    if 'distances' not in dados:
        raise RuntimeError(f"Erro na API do OpenRouteService: {dados}")

    registros = []
    for i in range(n):
        for j in range(n):
            if i != j:
                registros.append({
                    'id_cliente_origem': df_clientes.iloc[i]['id'],
                    'id_cliente_destino': df_clientes.iloc[j]['id'],
                    'distancia_km': dados['distances'][i][j] / 1000,
                })
    return pd.DataFrame(registros)


def calcular_melhor_rota(df_entregas_grupo, df_dist_clientes):
    """
    Ordena as entregas de um grupo (um armazém + uma data de saída) usando
    a heurística do vizinho mais próximo: parte do cliente mais perto do
    armazém e, a cada passo, segue para o cliente não visitado mais
    próximo do ponto atual.

    Não é a rota matematicamente ótima (isso seria um problema de TSP),
    mas é uma aproximação simples e rápida de calcular.

    Parâmetros
    ----------
    df_entregas_grupo : DataFrame
        Linhas de UM armazém + UMA data, com colunas id_cliente e
        distancia_km (armazém -> cliente).
    df_dist_clientes : DataFrame
        Resultado de calcula_distancias_clientes(), com a distância
        entre cada par de clientes.

    Retorno
    -------
    DataFrame com id_cliente, ordem_entrega (1, 2, 3...) e
    distancia_percorrida (distância do ponto anterior até aquela parada
    — não é acumulada).
    """
    # um cliente pode ter mais de uma venda no mesmo dia, mas só é
    # visitado uma vez fisicamente
    df_entregas_grupo = df_entregas_grupo.drop_duplicates(subset=['id_cliente'])

    clientes_restantes = df_entregas_grupo['id_cliente'].tolist()

    # ponto de partida: cliente mais próximo do armazém
    primeira_linha = df_entregas_grupo.sort_values('distancia_km').iloc[0]
    cliente_atual = primeira_linha['id_cliente']

    ordem = [cliente_atual]
    distancias = [primeira_linha['distancia_km']]
    clientes_restantes.remove(cliente_atual)

    # a cada passo, vai para o cliente restante mais próximo do atual
    while clientes_restantes:
        candidatos = df_dist_clientes[
            (df_dist_clientes['id_cliente_origem'] == cliente_atual) &
            (df_dist_clientes['id_cliente_destino'].isin(clientes_restantes))
        ]
        proxima_linha = candidatos.sort_values('distancia_km').iloc[0]

        ordem.append(proxima_linha['id_cliente_destino'])
        distancias.append(proxima_linha['distancia_km'])

        clientes_restantes.remove(proxima_linha['id_cliente_destino'])
        cliente_atual = proxima_linha['id_cliente_destino']

    return pd.DataFrame({
        'id_cliente': ordem,
        'ordem_entrega': range(1, len(ordem) + 1),
        'distancia_percorrida': distancias,
    })

In [0]:
"""
Script principal: gera a base sintética completa da transportadora
(homenagem a The Office) e calcula a rota otimizada de entrega para
cada armazém, em cada data de saída.
"""

# ---------------------------------------------------------------------
# Configuração
# ---------------------------------------------------------------------

load_dotenv()
api_key = os.getenv('ORS_API_KEY')

# período em que as vendas serão sorteadas, e quantidade de vendas a gerar
datas = pd.date_range(start="2026-01-01", end="2026-07-30")
n = 2000

# mostra todas as linhas ao exibir DataFrames grandes (sem truncar com "...")
pd.set_option('display.max_rows', None)


# ---------------------------------------------------------------------
# 1. Geração das entidades base
# ---------------------------------------------------------------------

# municípios e coordenadas são carregados uma única vez e reaproveitados
# por montar_armazens() e montar_clientes(), evitando repetir chamadas de API
df_municipios = ft.carrega_municipios()
df_lat_long = ft.carrega_lat_long()

df_armazem = ft.montar_armazens(df_municipios, df_lat_long)
df_produtos = ft.montar_produtos()
df_clientes = ft.montar_clientes(df_municipios, df_lat_long)
df_funcionarios = ft.monta_funcionarios()
df_vendas = ft.montar_vendas(df_produtos, df_clientes, df_funcionarios, datas, n)

# atribui a cada cliente o armazém mais próximo (distância real via OpenRouteService)
df_entrega = ft.atribui_armazem(df_armazem, df_clientes, api_key)


# ---------------------------------------------------------------------
# 2. Junta entregas com vendas para descobrir a data de saída de cada rota
# ---------------------------------------------------------------------

# cada venda gera uma entrega no dia seguinte à compra
df_entrega_completo = df_entrega.merge(
    df_vendas,
    how='inner',
    on=['id_cliente'],
)
df_entrega_completo['data_saida'] = df_entrega_completo['data_venda'] + pd.Timedelta(days=1)


# ---------------------------------------------------------------------
# 3. Roteirização: ordem de visita dos clientes por armazém + data
# ---------------------------------------------------------------------
"""
 Distância entre cada par de clientes, usada para montar a rota depois
 da primeira entrega (o caminho segue de cliente em cliente, não volta
 ao armazém a cada parada)
"""
df_dist_clientes = ft.calcula_distancias_clientes(df_clientes, api_key)

"""
 Calcula a melhor rota (vizinho mais próximo) separadamente para cada
 combinação de armazém + data de saída
"""
rotas_por_grupo = []
for (id_armazem, data_saida), grupo in df_entrega_completo.groupby(['id_armazem', 'data_saida']):
    rota = ft.calcular_melhor_rota(grupo, df_dist_clientes)
    rota['id_armazem'] = id_armazem
    rota['data_saida'] = data_saida
    rotas_por_grupo.append(rota)

df_rotas_otimizadas = pd.concat(rotas_por_grupo, ignore_index=True)
"""
 Ordena o resultado final: por data, depois por armazém, depois pela
 sequência de entrega dentro de cada rota
"""
df_rotas = df_rotas_otimizadas.sort_values(['data_saida', 'id_armazem', 'ordem_entrega'])

display(df_rotas)